# Interaction-Barcoded RL on Google Colab

This notebook adapts the REINVENT4 + GNINA + ProLIF + PoseBusters + P2Rank pipeline
(from the earlier design) to run inside Google Colab.

**Read this before running anything — set expectations first:**

- **Free-tier Colab (T4 GPU, idle/session timeouts, weak CPU) is realistic for
  Stage 1 and a limited Stage 2** (cheap filters + a modest amount of
  docking-based reward). A full multi-stage RL run with thousands of GNINA
  calls is a Colab Pro/Pro+ job at minimum, and may still want a real
  workstation or cluster for a production run. Treat a first Colab pass as a
  *pipeline shakedown*, not a final campaign.
- **Everything that must survive a session restart goes on Google Drive** —
  checkpoints, configs, receptor files, logs. Colab's local disk
  (`/content`) is wiped between sessions.
- **This notebook is a structural adaptation, not a guaranteed one-click
  run.** Package versions (REINVENT4/PyTorch/CUDA, GNINA's binary vs your
  GPU driver) can drift. Where a step is genuinely uncertain, that's called
  out explicitly rather than papered over.

Work through the cells top to bottom once; after that you can skip the
install cells on a warm session and jump straight to the run cells.


## 0. Check the runtime
Make sure **Runtime → Change runtime type → T4 GPU** (or better) is selected before running this cell.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Everything durable (receptor structures, pocket/fingerprint configs, REINVENT4
checkpoints, logs, GNINA binary) lives under a project folder on Drive so it
survives session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/ibrl_project"
for sub in ["configs", "logs", "checkpoints", "target", "bin"]:
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)
print("Project folder ready at", PROJECT)


## 2. Install conda via condacolab

REINVENT4 pins specific PyTorch/dependency versions that can clash with
whatever Colab's base Python environment already has installed, so we give it
an isolated conda-managed environment instead of installing into Colab's
system Python.

**This cell restarts the Colab kernel automatically.** That's expected —
wait for the restart, then continue with the next cell. Do not re-run this
cell after the restart.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


_(kernel restarts here — continue below once it's back)_

In [ ]:
import condacolab
condacolab.check()


## 3. Install REINVENT4

Note from the earlier design review: REINVENT4 maintainers have said
DockStream is no longer actively supported, so we don't install it — docking
is handled by GNINA directly, called from our own scoring script.

Check the `nvidia-smi` output from Step 0 for your CUDA driver version and
pick a matching install target below (`cu121`, `cu124`, `cu126`, etc. — see
`python install.py --help` after cloning). If you're unsure, `cpu` is the
safe fallback: per REINVENT4's own docs, a GPU matters most for transfer
learning, and reinforcement learning is less GPU-bound since most scoring
components (including ours) run on CPU anyway.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/MolecularAI/REINVENT4.git
%cd /content/REINVENT4
!python install.py --help


In [ ]:
# Pick ONE of these based on the --help output and your nvidia-smi CUDA version.
# Uncomment the line you want.

# !python install.py cpu
# !python install.py cu121
!python install.py cu124


In [ ]:
!reinvent --help


## 4. Install GNINA (docking + CNN affinity)

We pull the latest pre-built binary from GNINA's GitHub releases rather than
compiling from source (compiling needs CUDA toolkit + Boost + RDKit dev
headers, which is a lot of fragile setup for a Colab session).

GNINA's own README links an example Colab notebook, so this general approach
is a well-trodden path — but treat the exact release asset name as something
to verify: GitHub release asset naming has changed between versions (plain
build vs a separate cuda12.8 build for newer cards). If GPU CNN scoring
fails to run on Colab's driver, fall back to `--no_gpu` (still fully
functional, just slower).

In [ ]:
%cd /content
import json, urllib.request

api_url = "https://api.github.com/repos/gnina/gnina/releases/latest"
with urllib.request.urlopen(api_url) as resp:
    release = json.load(resp)

print("Latest GNINA release:", release["tag_name"])
for asset in release["assets"]:
    print(" -", asset["name"], asset["browser_download_url"])


In [ ]:
# After inspecting the asset list above, set this to the right download URL
# (prefer a plain linux binary asset; only pick a cuda12.8-specific one if
# your nvidia-smi CUDA version is 12.8+).
GNINA_URL = "SET_ME_FROM_THE_ASSET_LIST_ABOVE"

GNINA_BIN = "/content/drive/MyDrive/ibrl_project/bin/gnina"
!wget -q -O "{GNINA_BIN}" "{GNINA_URL}"
!chmod +x "{GNINA_BIN}"
!"{GNINA_BIN}" --version || echo "If this failed, re-check the asset URL above."


## 5. Install P2Rank (pocket detection)

P2Rank is a Java tool — install a JRE, then download and unpack its release
tarball once onto Drive so you don't re-download it every session.

In [ ]:
!apt-get -qq install -y default-jre-headless > /dev/null
!java -version


In [ ]:
%cd /content/drive/MyDrive/ibrl_project/bin
import json, urllib.request

api_url = "https://api.github.com/repos/rdk/p2rank/releases/latest"
with urllib.request.urlopen(api_url) as resp:
    release = json.load(resp)

print("Latest P2Rank release:", release["tag_name"])
for asset in release["assets"]:
    print(" -", asset["name"], asset["browser_download_url"])


In [ ]:
# Set this from the asset list above (the .tar.gz / .zip distribution asset).
P2RANK_URL = "SET_ME_FROM_THE_ASSET_LIST_ABOVE"

!wget -q -O p2rank.tar.gz "{P2RANK_URL}"
!tar -xzf p2rank.tar.gz
!ls


## 6. Install the scoring dependencies (ProLIF, MDAnalysis, PoseBusters, RDKit)

In [ ]:
!pip install -q rdkit prolif MDAnalysis posebusters


## 7. One-time target setup

Do this once per target, not per RL run. Upload your receptor structure
(`receptor.pdb`) to `ibrl_project/target/` in Drive first (drag-and-drop in
the Colab file browser, or `files.upload()`), then run P2Rank on it.

In [ ]:
RECEPTOR = "/content/drive/MyDrive/ibrl_project/target/receptor.pdb"
P2RANK_HOME = "/content/drive/MyDrive/ibrl_project/bin/p2rank_2.5"  # match the unpacked folder name from Step 5

!"{P2RANK_HOME}/prank" predict -f "{RECEPTOR}" -o /content/p2rank_out
!ls /content/p2rank_out


In [ ]:
# Inspect /content/p2rank_out/<receptor>_predictions.csv for the top pocket's
# center coordinates, then write pocket JSON by hand (or parse the CSV
# programmatically). Example:

import json

pocket = {
    "center_x": 0.0,   # <-- fill in from the P2Rank predictions CSV
    "center_y": 0.0,
    "center_z": 0.0,
    "size_x": 20,
    "size_y": 20,
    "size_z": 20,
}

with open("/content/drive/MyDrive/ibrl_project/configs/p2rank_pocket.json", "w") as f:
    json.dump(pocket, f, indent=2)
print("Wrote pocket config.")


**Ideal interaction fingerprint** — this is the manual/semi-manual step from
the earlier design: run PLIP or ProLIF on a *real* reference complex (a
co-crystallized ligand, or the docked pose of a known active) to list the
interactions you want reproduced, and save them as
`ibrl_project/configs/ideal_fingerprint.json` in the same
`{"interactions": [{"residue": ..., "type": ...}, ...]}` format used before.
There isn't a one-liner for this — it's a judgment call about which
interactions actually matter for this target.

## 8. Write the REINVENT4 config and scoring script to Drive

Same two files as the earlier design, with paths pointed at this Colab
session's Drive layout and the GNINA binary installed above.

In [ ]:
toml_content = """
run_type = "staged_learning"
device = "cuda:0"
tb_logdir = "/content/drive/MyDrive/ibrl_project/logs/tb_ibrl"
json_out_config = "/content/drive/MyDrive/ibrl_project/logs/_ibrl_staged_learning.json"

[parameters]
summary_csv_prefix = "/content/drive/MyDrive/ibrl_project/logs/ibrl"
use_checkpoint = false
purge_memories = false
prior_file = "/content/REINVENT4/priors/reinvent.prior"
agent_file = "/content/REINVENT4/priors/reinvent.prior"
batch_size = 32

[learning_strategy]
type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]
type = "IdenticalMurckoScaffold"
bucket_size = 25
minscore = 0.4

[[stage]]
chkpt_file = "/content/drive/MyDrive/ibrl_project/checkpoints/stage1.chkpt"
termination = "simple"
max_steps = 100
min_steps = 25

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.QED]
[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 1.0

[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]
[[stage.scoring.component.MolecularWeight.endpoint]]
name = "MW"
weight = 0.5
transform.type = "double_sigmoid"
transform.high = 500
transform.low = 200

[[stage]]
chkpt_file = "/content/drive/MyDrive/ibrl_project/checkpoints/stage2.chkpt"
termination = "simple"
max_steps = 150
min_steps = 50

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.QED]
[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 0.5

[[stage.scoring.component]]
[stage.scoring.component.ExternalProcess]
[[stage.scoring.component.ExternalProcess.endpoint]]
name = "InteractionBarcode"
weight = 1.5
params.executable = "/usr/local/bin/python3"
params.args = "/content/drive/MyDrive/ibrl_project/configs/structure_barcode_score.py --target-pdb /content/drive/MyDrive/ibrl_project/target/receptor.pdb --pocket-config /content/drive/MyDrive/ibrl_project/configs/p2rank_pocket.json --ideal-fp /content/drive/MyDrive/ibrl_project/configs/ideal_fingerprint.json --gnina-bin /content/drive/MyDrive/ibrl_project/bin/gnina"
transform.type = "no_transformation"
"""

# NOTE: batch_size and max_steps are deliberately smaller here than a
# workstation run (32 / 100-150) to keep each stage inside a single free-tier
# Colab session. Raise them once you've confirmed the pipeline works and know
# your session budget (Colab Pro/Pro+, or checkpoint-resume across sessions
# per the note in Step 10 below).

with open("/content/drive/MyDrive/ibrl_project/configs/staged_learning.toml", "w") as f:
    f.write(toml_content)
print("Wrote staged_learning.toml")


In [ ]:
# structure_barcode_score.py — same scoring logic as the earlier design
# (embed -> GNINA dock -> PoseBusters gate -> dense ProLIF interaction
# recovery + GNINA CNN affinity). Reminder from before: the stdin format
# REINVENT4 passes to ExternalProcess scripts should be checked against
# support/run-chemprop.py in your cloned REINVENT4 repo if scores come back
# as all-zero — only the OUTPUT contract (below) is solidly confirmed.

script_content = r"""#!/usr/bin/env python3
import argparse, json, subprocess, sys, tempfile
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem


def read_smiles_from_stdin():
    raw = sys.stdin.read()
    try:
        payload = json.loads(raw)
        for key in ("smiles", "SMILES", "smilies"):
            if isinstance(payload, dict) and key in payload:
                return list(payload[key])
            if isinstance(payload, dict) and "payload" in payload and key in payload["payload"]:
                return list(payload["payload"][key])
        if isinstance(payload, list):
            return list(payload)
    except json.JSONDecodeError:
        pass
    return [line.strip() for line in raw.splitlines() if line.strip()]


def write_scores_to_stdout(scores):
    print(json.dumps({"version": 1, "payload": {"predictions": scores}}))


def embed_3d(smiles, workdir):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
    out_path = workdir / "ligand.sdf"
    w = Chem.SDWriter(str(out_path))
    w.write(mol)
    w.close()
    return out_path


def run_gnina(gnina_bin, receptor_pdb, ligand_sdf, pocket, workdir):
    out_path = workdir / "docked.sdf"
    cmd = [gnina_bin, "-r", receptor_pdb, "-l", str(ligand_sdf), "-o", str(out_path),
           "--center_x", str(pocket["center_x"]), "--center_y", str(pocket["center_y"]),
           "--center_z", str(pocket["center_z"]), "--size_x", str(pocket["size_x"]),
           "--size_y", str(pocket["size_y"]), "--size_z", str(pocket["size_z"]),
           "--cnn_scoring", "rescore", "--num_modes", "1", "--seed", "42"]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    except subprocess.TimeoutExpired:
        return None, 0.0
    if proc.returncode != 0 or not out_path.exists():
        return None, 0.0
    affinity = 0.0
    try:
        supplier = Chem.SDMolSupplier(str(out_path))
        top = supplier[0]
        if top is not None and top.HasProp("CNNaffinity"):
            affinity = float(top.GetProp("CNNaffinity"))
    except Exception:
        pass
    return out_path, affinity


def passes_posebusters(docked_sdf, receptor_pdb):
    try:
        from posebusters import PoseBusters
        buster = PoseBusters(config="dock")
        report = buster.bust([str(docked_sdf)], None, receptor_pdb)
        return bool(report.all(axis=1).iloc[0])
    except Exception:
        return False


def interaction_recovery_score(docked_sdf, receptor_pdb, ideal_fp):
    try:
        import MDAnalysis as mda
        import prolif as plf
        protein_u = mda.Universe(receptor_pdb)
        ligand_u = mda.Universe(str(docked_sdf))
        protein_mol = plf.Molecule.from_mda(protein_u)
        ligand_mol = plf.Molecule.from_mda(ligand_u)
        fp = plf.Fingerprint()
        fp.run_from_iterable([ligand_mol], protein_mol)
        df = fp.to_dataframe()
    except Exception:
        return 0.0
    if df.empty:
        return 0.0
    targets = ideal_fp.get("interactions", [])
    if not targets:
        return 0.0
    observed = {f"{c[1]}.{c[2]}" for c in df.columns}
    hits = sum(1 for t in targets if f"{t['residue']}.{t['type']}" in observed)
    return hits / len(targets)


def score_molecule(smiles, receptor_pdb, pocket, ideal_fp, gnina_bin):
    with tempfile.TemporaryDirectory() as tmp:
        workdir = Path(tmp)
        ligand_sdf = embed_3d(smiles, workdir)
        if ligand_sdf is None:
            return 0.0
        docked_sdf, cnn_affinity = run_gnina(gnina_bin, receptor_pdb, ligand_sdf, pocket, workdir)
        if docked_sdf is None:
            return 0.0
        if not passes_posebusters(docked_sdf, receptor_pdb):
            return 0.0
        interaction_score = interaction_recovery_score(docked_sdf, receptor_pdb, ideal_fp)
        affinity_score = max(0.0, min(cnn_affinity, 10.0)) / 10.0
        return 0.6 * interaction_score + 0.4 * affinity_score


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--target-pdb", required=True)
    parser.add_argument("--pocket-config", required=True)
    parser.add_argument("--ideal-fp", required=True)
    parser.add_argument("--gnina-bin", required=True)
    args = parser.parse_args()
    pocket = json.loads(Path(args.pocket_config).read_text())
    ideal_fp = json.loads(Path(args.ideal_fp).read_text())
    smiles_list = read_smiles_from_stdin()
    scores = [score_molecule(s, args.target_pdb, pocket, ideal_fp, args.gnina_bin) for s in smiles_list]
    write_scores_to_stdout(scores)


if __name__ == "__main__":
    main()
"""

with open("/content/drive/MyDrive/ibrl_project/configs/structure_barcode_score.py", "w") as f:
    f.write(script_content)
print("Wrote structure_barcode_score.py")


## 9. Sanity check in `scoring` mode

Run on a handful of known SMILES before committing to a full RL run — confirm
the scores look sane (a known active should score higher than a random
decoy) and that no path is broken.

In [ ]:
smiles_check = "/content/smiles_check.smi"
with open(smiles_check, "w") as f:
    f.write("CCO\n")           # ethanol — should dock/score poorly, sanity floor
    f.write("c1ccccc1\n")       # benzene — same idea

scoring_toml = f"""
run_type = "scoring"
json_out_config = "/content/drive/MyDrive/ibrl_project/logs/_scoring_check.json"

[parameters]
smiles_file = "{smiles_check}"
output_csv = "/content/drive/MyDrive/ibrl_project/logs/scoring_check.csv"

[scoring]
type = "geometric_mean"

[[scoring.component]]
[scoring.component.ExternalProcess]
[[scoring.component.ExternalProcess.endpoint]]
name = "InteractionBarcode"
weight = 1
params.executable = "/usr/local/bin/python3"
params.args = "/content/drive/MyDrive/ibrl_project/configs/structure_barcode_score.py --target-pdb /content/drive/MyDrive/ibrl_project/target/receptor.pdb --pocket-config /content/drive/MyDrive/ibrl_project/configs/p2rank_pocket.json --ideal-fp /content/drive/MyDrive/ibrl_project/configs/ideal_fingerprint.json --gnina-bin /content/drive/MyDrive/ibrl_project/bin/gnina"
"""

with open("/content/drive/MyDrive/ibrl_project/configs/scoring_check.toml", "w") as f:
    f.write(scoring_toml)

!reinvent -l /content/drive/MyDrive/ibrl_project/logs/scoring_check.log /content/drive/MyDrive/ibrl_project/configs/scoring_check.toml
!cat /content/drive/MyDrive/ibrl_project/logs/scoring_check.csv


## 10. Run staged learning — and how to survive Colab's session limits

Colab free tier disconnects on idle and caps total session length; Pro/Pro+
extends but doesn't remove this. The pattern that works across restarts:

1. Run a stage with `use_checkpoint = false` the first time.
2. If the session dies mid-stage, set `use_checkpoint = true` in the TOML
   and re-run — REINVENT4 will resume from the `chkpt_file` on Drive rather
   than starting over.
3. Keep `max_steps`/`batch_size` modest (as in the config above) so a stage
   is more likely to finish inside one session in the first place, rather
   than relying on resume every time.

In [ ]:
!reinvent -l /content/drive/MyDrive/ibrl_project/logs/staged_learning.log /content/drive/MyDrive/ibrl_project/configs/staged_learning.toml


## Notes / known gaps

- **GNINA docking speed is the real bottleneck on Colab** — free-tier Colab
  CPUs are weak (typically 2 shared cores), and GNINA's Vina-style pose
  sampling runs on CPU regardless of GPU CNN scoring. If Stage 2 is too slow,
  lower `batch_size` further, or reduce GNINA's exhaustiveness before
  reaching for more compute.
- **Cost control (MolPAL-style surrogate) and Boltz-2 reranking** from the
  earlier design notes are not implemented in this Colab version either —
  they remain the next steps once this pipeline runs end-to-end here.
- If a `reinvent` run reports scores of exactly 0.0 for everything, check
  `structure_barcode_score.py`'s stdin-parsing assumption first (flagged
  above) before assuming the chemistry is wrong.
